# Pipeline Triple MRD — FlowSOM × 3 Méthodes

## Objectif
Calculer la MRD (Maladie Résiduelle Détectable) en LAM/AML à partir d'un ou plusieurs fichiers FCS pathologiques, en appliquant **3 méthodes en parallèle** sur les mêmes données mappées sur le MST de référence NBM :

| Méthode | Logique | Référence |
|---------|---------|-----------|
| **M1** — Δ Métaclusters | Proportion patient > 2× NBM par métacluster | ELN 2022 (fold-change) |
| **M2** — Distance MST | Nœuds SOM déviants (distance euclidienne patient ↔ NBM > μ + k·σ) | Approche distancielle |
| **M3** — Mapping Pop. | Nœuds « Unknown » après mapping M12_cosine_prior (Section 10) | Pipeline principal |

## Architecture du module `mrd_pipeline.py`
```
NBMReference.build()           ← construire le modèle NBM (1 seule fois)
  ↓
run_mrd_pipeline_for_patient() ← 1 patient = 1 FCS
  ├── preprocess_fcs()          ← gating + transformation + alignement marqueurs
  ├── map_patient_to_nbm()      ← mapping SOM patient → nœuds NBM
  ├── mrd_method_1()            ← Δ métaclusters
  ├── mrd_method_2()            ← distances euclidiennes
  ├── mrd_method_3()            ← mapping populations
  └── select_best_mrd_method()  ← sélection + flag divergence
  ↓
run_mrd_batch()                ← cohorte entière, sauvegarde HTML + JSON + CSV
```

## Hypothèses de nommage
- Les **FCS NBM** sont dans un dossier dédié (`HEALTHY_FOLDER`).
- Les **CSV de référence MFI** (Méthode 3) sont dans `Data/Ref MFI/`.
- Les **FCS patients** sont dans `PATHOLOGICAL_FOLDER` (1 fichier = 1 patient).
- Si le modèle NBM a **déjà été calculé** dans le notebook principal, utiliser `NBMReference.build_from_precomputed()` pour éviter de le recalculer.

---
*Auteur : Florian Magne · Version 1.0 · Mars 2026*

In [ ]:
## Cellule 1 — Vérification des dépendances
import importlib.util

REQUIRED = ["flowsom", "anndata", "flowio", "numpy", "pandas", "scipy", "sklearn", "matplotlib"]
OPTIONAL  = ["flowkit", "fcswrite"]

print("=== Dépendances requises ===")
for pkg in REQUIRED:
    spec = importlib.util.find_spec(pkg.replace("-", "_"))
    status = "OK" if spec else "MANQUANT"
    print(f"  {pkg:<20} {status}")

print("\n=== Dépendances optionnelles ===")
for pkg in OPTIONAL:
    spec = importlib.util.find_spec(pkg.replace("-", "_"))
    status = "OK" if spec else "non installe"
    print(f"  {pkg:<20} {status}")

In [ ]:
## Cellule 2 — Imports + chargement du module mrd_pipeline

import sys
import os
from pathlib import Path

# ---- Chemin vers le module mrd_pipeline ----
MODULE_DIR = Path(r"C:\Users\Florian Travail\Documents\FlowSom")
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from mrd_pipeline import (
    DEFAULT_PARAMS,
    NBMReference,
    preprocess_fcs,
    mrd_method_1,
    mrd_method_2,
    mrd_method_3,
    select_best_mrd_method,
    run_mrd_pipeline_for_patient,
    run_mrd_batch,
    export_patient_json,
    export_patient_html,
    export_kaluza_csv,
)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("inline")
import matplotlib.pyplot as plt

print("Module mrd_pipeline charge avec succes.")
print(f"Parametres par defaut disponibles : {list(DEFAULT_PARAMS.keys())}")

In [ ]:
## Cellule 3 — Configuration des chemins et paramètres
#
# Utilise DEFAULT_PARAMS comme base et surcharge uniquement ce qui diffère.
# TOUS les paramètres du panneau de configuration (widgets FlowSOM_Analysis_Pipeline)
# sont disponibles via DEFAULT_PARAMS et peuvent être surchargés ici.

# ============================================================
# ADAPTER À VOTRE ENVIRONNEMENT
# ============================================================

# Dossier contenant les FCS normaux (NBM = Normal Bone Marrow)
NBM_FOLDER = Path(r"C:\Users\Florian Travail\Documents\FlowSom\Data\Moelle normale")

# Dossier contenant les FCS pathologiques patients
PATHOLOGICAL_FOLDER = Path(r"C:\Users\Florian Travail\Documents\FlowSom\Data\Patho")

# Dossier CSV de référence MFI populations (optionnel, pour Méthode 3)
REF_MFI_CSV_FOLDER = Path(r"C:\Users\Florian Travail\Documents\FlowSom\Data\Ref MFI")

# Dossier de sortie pour les rapports HTML/JSON/CSV
OUTPUT_DIR = Path(r"C:\Users\Florian Travail\Documents\FlowSom\output\mrd_reports")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# PARAMÈTRES DU PIPELINE — Extension de DEFAULT_PARAMS
# ============================================================
PARAMS = DEFAULT_PARAMS.copy()
PARAMS.update({
    # ── Section 1 : Chemins ─────────────────────────────────
    "healthy_folder": NBM_FOLDER,
    "pathological_folder": PATHOLOGICAL_FOLDER,
    "compare_mode": True,

    # ── Section 2 : Pré-gating ──────────────────────────────
    "apply_pregating": True,
    "gating_mode": "auto",          # "auto" (GMM/RANSAC) | "manual" (percentiles)
    "mode_blastes": True,

    # Gate 1 — Débris (FSC/SSC)
    "gate_debris": True,
    "debris_min_pct": 1.0,
    "debris_max_pct": 99.0,

    # Gate 2 — Doublets (FSC-A/FSC-H)
    "gate_doublets": True,
    "ratio_min": 0.6,
    "ratio_max": 1.4,

    # Gate 3 — CD45+ leucocytes
    "gate_cd45": True,
    "cd45_pct": 5,

    # Gate 4 — Blastes CD34+ (optionnel)
    "filter_blasts": False,
    "cd34_pct": 85,
    "ssc_filter": True,
    "ssc_max_pct": 70,

    # ── Section 3 : Transformation ──────────────────────────
    "transform": "logicle",         # "logicle" | "arcsinh" | "log10" | "none"
    "cofactor": 5.0,
    "apply_to_scatter": False,

    # ── Section 4 : Sélection marqueurs ─────────────────────
    "exclude_scatter": True,
    "exclude_markers": ["CD45"],

    # ── Section 4b : Filtrage -A / -H (Area vs Height) ──────
    # Identique à APPLY_MARKER_FILTERING / KEEP_AREA / KEEP_HEIGHT
    # du notebook FlowSOM_Analysis_Pipeline.ipynb (Section 8).
    #   -A (Area)   = aire sous la courbe du pulse → plus stable → RECOMMANDÉ
    #   -H (Height) = hauteur maximale du pulse    → plus sensible
    # Mettre keep_area=False + keep_height=True pour utiliser les canaux -H à la place.
    "apply_marker_filtering": True,  # True = dédupliquer -A et -H (garder un seul type)
    "keep_area":   True,             # True = conserver les canaux -A [RECOMMANDÉ]
    "keep_height": False,            # True = conserver les canaux -H

    # ── Section 5 : FlowSOM ─────────────────────────────────
    "xdim": 10,
    "ydim": 10,
    "rlen": "auto",                 # "auto" = √N × 0.1; ou entier fixe (10, 20, 50, 100)
    "n_clusters": 7,                # Métaclusters (ELN recommande 7-10 pour LAM)
    "seed": 42,

    # Auto-clustering (Stabilité ARI + Silhouette — Méthode 2024)
    "auto_cluster": False,          # True → optimise k automatiquement
    "min_k": 5,
    "max_k": 35,
    "n_bootstrap": 10,
    "sample_boot": 20000,
    "min_stability": 0.75,          # Seuil ARI minimal
    "w_stability": 0.65,            # Poids stabilité (w_stability + w_silhouette = 1.0)
    "w_silhouette": 0.35,

    # ── AutoGating avancé ────────────────────────────────────
    "gmm_max_samples": 200_000,     # Sous-échantillonnage avant GMM
    "ransac_r2_threshold": 0.85,    # Seuil R² RANSAC (< → fallback ratio)

    # ── MRD méthode 1 (Δ Métaclusters — ELN 2022) ───────────
    "mrd1_fold_change": 2.0,        # Seuil ELN : patient/NBM > 2×
    "mrd1_min_events": 17,          # Minimum d'événements ELN par nœud

    # ── MRD méthode 2 (Distance euclidienne cell-level MAD) ──
    "mad_allowed": 4,               # MADs au-dessus de la médiane NBM (ELN 2022)
    "mrd2_ratio_threshold": 2.0,
    "mrd2_min_cells_per_cluster": 10,

    # ── MRD méthode 3 (benchmark M3/M8/M9/M12) ──────────────
    "mrd3_mapping_method": "M12_cosine_prior",
    "mrd3_unknown_mode": "auto_otsu",
    "mrd3_hard_limit_factor": 5.0,
    "m3_benchmark_methods": ["M3_cosine", "M8_ref_norm", "M9_prior", "M12_cosine_prior"],
    "m3_log_benchmark": True,

    # ── Output ───────────────────────────────────────────────
    "output_dir": str(OUTPUT_DIR),
})

# ── Vérification des dossiers ────────────────────────────────────────────────
for name, path in [("NBM", NBM_FOLDER), ("Patient", PATHOLOGICAL_FOLDER)]:
    exists = path.exists()
    fcs_count = len(list(path.glob("*.fcs"))) if exists else 0
    status = f"{fcs_count} fichiers FCS" if exists else "DOSSIER INTROUVABLE"
    print(f"  {name:<12} {path.name:<40} {status}")

ref_exists = REF_MFI_CSV_FOLDER.exists()
csv_count = len(list(REF_MFI_CSV_FOLDER.glob("*.csv"))) if ref_exists else 0
print(f"  {'Ref MFI':<12} {REF_MFI_CSV_FOLDER.name:<40} {csv_count} fichiers CSV")

suffix_label = ("-A seuls" if PARAMS["keep_area"] and not PARAMS["keep_height"]
                else "-H seuls" if PARAMS["keep_height"] and not PARAMS["keep_area"]
                else "-A et -H" if PARAMS["keep_area"] and PARAMS["keep_height"]
                else "désactivé")
print(f"\n  Mode gating    : {PARAMS['gating_mode'].upper()}")
print(f"  Transformation : {PARAMS['transform'].upper()}")
print(f"  Filtrage -A/-H : {'actif → ' + suffix_label if PARAMS['apply_marker_filtering'] else 'désactivé (tous marqueurs conservés)'}")
print(f"  FlowSOM        : {PARAMS['xdim']}×{PARAMS['ydim']} = {PARAMS['xdim']*PARAMS['ydim']} nœuds")
print(f"  rlen           : {PARAMS['rlen']}")
print(f"  k métaclusters : {PARAMS['n_clusters']} (auto={PARAMS['auto_cluster']})")
print(f"  Seed           : {PARAMS['seed']}")
print(f"\n  PARAMS disponibles : {len(PARAMS)} clés")


## Section 1 — Construction du modèle de référence NBM

Le modèle NBM est construit **une seule fois** à partir des FCS de moelles osseuses normales (`Data/PoolNBM`).

Il encapsule :
- Le **MST FlowSOM** de référence (nœuds SOM + métaclusters)
- Les **profils d'expression MFI** par nœud (centroïdes des poids SOM)
- Les **proportions de métaclusters** en NBM (référence fold-change Méthode 1)
- Les **distances inter-nœuds** en NBM (référence seuil Méthode 2)
- Le **dictionnaire de populations** annotées (Méthode 3, optionnel)

> **Note :** Si le modèle FlowSOM a déjà été calculé dans `FlowSOM_Analysis_Pipeline.ipynb`, il est possible de l'injecter directement via `build_from_precomputed()` (voir cellule suivante). Cela évite de relancer le SOM sur les NBM.

In [ ]:
## Cellule 4a — Construction du modele NBM depuis les FCS (methode standard)

# Construction complete depuis les FCS bruts (recommande si NBM non encore calcule)
nbm = NBMReference(
    nbm_folder=NBM_FOLDER,
    params=PARAMS,
)

nbm.build(
    ref_mfi_csv_folder=REF_MFI_CSV_FOLDER if REF_MFI_CSV_FOLDER.exists() else None,
    verbose=True,
)

print("\n=== Modele NBM construit ===")
print(f"  Marqueurs      : {len(nbm.markers)}")
print(f"  Noeuds SOM     : {nbm.node_mfi.shape[0]}")
print(f"  Metaclusters   : {nbm.n_metaclusters}")
print(f"  Total cellules : {nbm.total_cells:,}")
if nbm.pop_mfi_ref is not None:
    print(f"  Pop. ref MFI   : {list(nbm.pop_mfi_ref.index)}")

In [ ]:
## Cellule 4b — ALTERNATIVE : Injection d'un modele NBM precalcule
#
# Si vous avez deja execute la Section 10 de FlowSOM_Analysis_Pipeline.ipynb,
# utilisez cette cellule PLUTOT que 4a pour eviter de recalculer le SOM.
#
# Variables attendues depuis le notebook principal (Section 10.2) :
#   node_mfi_raw_df     : DataFrame (n_nodes x n_markers) — profils d'expression SOM
#   _node_count_raw     : array(n_nodes,) — nb de cellules NBM par noeud
#   _mc_per_node_raw    : Series(n_nodes,) — numero de metacluster par noeud
#   df_mfi_raw_ref      : DataFrame — MFI de reference par population (optionnel)
#   pop_cell_counts     : dict — nb de cellules par population (optionnel)
#
# DECOMMENTER ET ADAPTER SI VOUS AVEZ CES VARIABLES DISPONIBLES :

# nbm = NBMReference(nbm_folder=NBM_FOLDER, params=PARAMS)
# nbm.build_from_precomputed(
#     node_mfi    = node_mfi_raw_df.values,           # Section 10.2
#     node_counts = _node_count_raw,                   # Section 10.2
#     metaclusters= _mc_per_node_raw.values,           # Section 10.2
#     markers     = list(node_mfi_raw_df.columns),
# )
# # Optionnel : injecter les references de populations pour la Methode 3
# nbm.pop_mfi_ref     = df_mfi_raw_ref                # Section 10.3
# nbm.pop_cell_counts = pop_cell_counts               # Section 10.3b
#
# print("Modele NBM injecte depuis notebook principal.")
# print(f"  Marqueurs : {len(nbm.markers)} | Noeuds : {nbm.node_mfi.shape[0]}")

print("Cellule 4b en commentaire — decommenter si modele precalcule disponible.")

## Section 2 — Analyse d'un patient unique (diagnostic/suivi)

Exécuter le pipeline complet sur **un seul FCS** pour vérifier la configuration et inspecter les résultats détaillés avant de lancer le batch.

**Sortie** : dict structuré avec :
- `patient_id`, `fcs_path`, `n_cells`
- `mrd.method_1` / `mrd.method_2` / `mrd.method_3` — résultats détaillés de chaque méthode
- `mrd.best_method`, `mrd.confidence`, `mrd.divergence_flag`
- `kaluza_table` — table de gating compatible Kaluza (par nœud SOM)
- `markers`, `preprocessing_qc`

In [ ]:
## Cellule 5 — Analyse patient unique

# Selectionner le FCS a analyser
fcs_files_patient = sorted(PATHOLOGICAL_FOLDER.glob("*.fcs"))
if not fcs_files_patient:
    raise FileNotFoundError(f"Aucun fichier FCS dans : {PATHOLOGICAL_FOLDER}")

TEST_FCS = fcs_files_patient[0]   # Premier patient par defaut (modifier l'indice)
print(f"FCS analyse : {TEST_FCS.name}")

# Lancer le pipeline complet (M1 + M2 + M3 en parallele interne)
result = run_mrd_pipeline_for_patient(
    fcs_path   = TEST_FCS,
    nbm        = nbm,
    params     = PARAMS,
    patient_id = TEST_FCS.stem,
    verbose    = True,
)

# Affichage synthetique
mrd = result["mrd"]
if result.get("errors"):
    print(f"\n  Avertissements pipeline : {result['errors']}")

print("\n" + "="*55)
print(f"  Patient     : {result['patient_id']}")
print(f"  Cellules    : {result['n_cells']:,}")
print(f"  Marqueurs   : {len(result['markers'])}")
print("="*55)
print(f"  MRD Methode 1 (delta MC)     : {mrd['method_1'].get('mrd_percent', mrd['method_1'].get('mrd_value', 0.0) * 100):.4f}%")
print(f"  MRD Methode 2 (distance MST) : {mrd['method_2'].get('mrd_percent', mrd['method_2'].get('mrd_value', 0.0) * 100):.4f}%")
print(f"  MRD Methode 3 (mapping pop)  : {mrd['method_3'].get('mrd_percent', mrd['method_3'].get('mrd_value', 0.0) * 100):.4f}%")
for m_key in ("method_1", "method_2", "method_3"):
    if "error" in mrd.get(m_key, {}):
        print(f"  !! Erreur {m_key} : {mrd[m_key]['error']}")
print(f"  Methode retenue              : {mrd.get('best_method', 'N/A')}")
print(f"  Confiance                    : {mrd.get('confidence', 0.0):.2f}")
if mrd.get("divergence_flag"):
    print("  ATTENTION : divergence inter-methodes detectee !")
print("="*55)


In [ ]:
## Cellule 6 — Export des rapports patient (JSON + HTML + CSV Kaluza)

import json

# Export JSON
json_path = export_patient_json(result, OUTPUT_DIR)
print(f"JSON sauvegarde : {json_path}")

# Export HTML (rapport complet avec graphe et table Kaluza)
html_path = export_patient_html(result, OUTPUT_DIR)
print(f"HTML sauvegarde : {html_path}")

# Export CSV Kaluza (table par noeud SOM, compatible Kaluza Beckman Coulter)
csv_path = export_kaluza_csv(result, OUTPUT_DIR)
print(f"CSV Kaluza sauvegarde : {csv_path}")

# Apercu de la table Kaluza
kaluza_df = pd.DataFrame(result["kaluza_table"])
print(f"\nTable Kaluza : {len(kaluza_df)} noeuds x {len(kaluza_df.columns)} colonnes")
display(kaluza_df.head(10))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle(f"MRD Triple — Patient : {result['patient_id']}", fontsize=13, fontweight="bold")

# --- Graphe 1 : comparaison des 3 methodes ---
methods = ["M1: delta MC", "M2: new_data", "M3: mapping"]
values  = [
    mrd["method_1"].get("mrd_percent", 0.0),
    mrd["method_2"].get("mrd_percent", 0.0),
    mrd["method_3"].get("mrd_percent", 0.0),
]
colors = ["#4f86c6", "#e88a3e", "#5caa5e"]
bars = axes[0].bar(methods, values, color=colors, edgecolor="white", linewidth=0.8)
axes[0].axhline(0.005, color="red",    linestyle="--", linewidth=1.2, label="LOQ 0.005%")
axes[0].axhline(0.009, color="orange", linestyle=":",  linewidth=1.2, label="LOD 0.009%")
axes[0].set_title("MRD par methode (%)")
axes[0].set_ylabel("MRD (%)")
axes[0].legend(fontsize=8)
for bar, val in zip(bars, values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0001,
                 f"{val:.4f}%", ha="center", va="bottom", fontsize=8)

# --- Graphe 2 : fold-change par metacluster (Methode 1 — arbre mixte) ---
mc_fc = mrd["method_1"].get("mc_fold_changes", {})
if mc_fc:
    mc_labels = [f"MC{k}" for k in sorted(mc_fc.keys())]
    mc_vals   = [mc_fc[k] for k in sorted(mc_fc.keys())]
    bar_colors_mc = ["#e05555" if v >= PARAMS["mrd1_fold_change"] else "#5a8fd4" for v in mc_vals]
    axes[1].bar(mc_labels, mc_vals, color=bar_colors_mc, edgecolor="white")
    axes[1].axhline(PARAMS["mrd1_fold_change"], color="red", linestyle="--",
                    linewidth=1.2, label=f"seuil x{PARAMS['mrd1_fold_change']}")
    axes[1].set_title("M1 mixte : Fold-change patient/NBM")
    axes[1].set_ylabel("Fold-change")
    axes[1].legend(fontsize=8)
else:
    axes[1].text(0.5, 0.5, "M1 : donnees\nnon disponibles", ha="center", va="center",
                 transform=axes[1].transAxes)
    axes[1].set_title("M1 : Fold-change")

# --- Graphe 3 : statistiques distances cellulaires (Methode 2 — new_data) ---
dist_stats = mrd["method_2"].get("cell_distances_stats", {})
n_mrd_c    = mrd["method_2"].get("n_mrd_cells", 0)
n_total_c  = mrd["method_2"].get("n_cells_patient", 1)
if dist_stats:
    stat_labels = ["Mediane", "Moyenne", "Max"]
    stat_vals   = [dist_stats.get("median", 0), dist_stats.get("mean", 0), dist_stats.get("max", 0)]
    bar_c = axes[2].bar(stat_labels, stat_vals, color=["#8a5692", "#b07abc", "#d9aadf"],
                        edgecolor="white")
    for b, v in zip(bar_c, stat_vals):
        axes[2].text(b.get_x() + b.get_width()/2, b.get_height() * 1.01,
                     f"{v:.3f}", ha="center", va="bottom", fontsize=8)
    mad_a = mrd["method_2"].get("mad_allowed", "?")
    axes[2].set_title(f"M2 new_data — stats distances\n"
                      f"({n_mrd_c}/{n_total_c} cellules MRD, MAD={mad_a}x)")
    axes[2].set_ylabel("Distance euclidienne")
else:
    axes[2].text(0.5, 0.5, "M2 : donnees\nnon disponibles", ha="center", va="center",
                 transform=axes[2].transAxes)
    axes[2].set_title("M2 : Distances cellulaires")

# --- Graphe 4 : repartition cellules M3 (Unknown vs Normal) ---
n_unknown = mrd["method_3"].get("n_unknown_cells", 0)
n_normal  = result["n_cells"] - n_unknown
labels_m3 = ["Blastes / Unknown", "Normal"]
sizes_m3  = [max(n_unknown, 0), max(n_normal, 0)]
colors_m3 = ["#e05555", "#5caa5e"]
if sum(sizes_m3) > 0:
    axes[3].pie(sizes_m3, labels=labels_m3, colors=colors_m3,
                autopct="%1.2f%%", startangle=90, textprops={"fontsize": 9})
axes[3].set_title("M3 mixte : Repartition cellules")

plt.tight_layout()
plt.show()

## Section 3 — Traitement de cohorte (batch)

Traitement de **tous les FCS patients** du dossier `PATHOLOGICAL_FOLDER` en série.

Pour chaque patient :
1. Prétraitement (gating + transformation)
2. Mapping sur le MST NBM
3. Calcul M1 + M2 + M3 en parallèle interne
4. Sélection de la meilleure méthode
5. Export HTML + JSON + CSV Kaluza

**Résultat** : `cohort_df` — DataFrame de synthèse avec une ligne par patient.

In [ ]:
## Cellule 8 — Lancement du batch cohorte

fcs_files_batch = sorted(PATHOLOGICAL_FOLDER.glob("*.fcs"))
print(f"Patients a analyser : {len(fcs_files_batch)}")
for f in fcs_files_batch:
    print(f"  {f.name}")

if not fcs_files_batch:
    raise FileNotFoundError(f"Aucun FCS trouve dans : {PATHOLOGICAL_FOLDER}")

# Lance l'analyse de toute la cohorte
# verbose=True affiche la progression patient par patient
cohort_df = run_mrd_batch(
    fcs_file_list = fcs_files_batch,
    nbm           = nbm,
    params        = PARAMS,
    output_dir    = OUTPUT_DIR,
    verbose       = True,
)

print(f"\nCohorte terminee : {len(cohort_df)} patients analyses.")
print(f"Rapports sauvegardes dans : {OUTPUT_DIR}")

In [ ]:
## Cellule 9 — Tableau de synthese de la cohorte

# Affichage du tableau complet avec formatage
# Note : .applymap() renomme en .map() depuis pandas 2.1
_styler = (cohort_df.style
    .format({
        "MRD_method_1_pct": "{:.4f}%",
        "MRD_method_2_pct": "{:.4f}%",
        "MRD_method_3_pct": "{:.4f}%",
        "n_cells": "{:,}",
        "confidence": "{:.2f}",
    })
    .background_gradient(subset=["MRD_method_1_pct","MRD_method_2_pct","MRD_method_3_pct"],
                          cmap="YlOrRd", vmin=0, vmax=1.0)
    .set_caption("Cohorte MRD Triple — Recapitulatif")
)
if "divergence_flag" in cohort_df.columns:
    _flag_fn = lambda v: "background-color: #ffe0e0" if v else ""
    try:
        _styler = _styler.map(_flag_fn, subset=["divergence_flag"])       # pandas >= 2.1
    except AttributeError:
        _styler = _styler.applymap(_flag_fn, subset=["divergence_flag"])  # pandas < 2.1
display(_styler)

# Export CSV recapitulatif de la cohorte
cohort_csv_path = OUTPUT_DIR / "cohort_summary.csv"
cohort_df.to_csv(cohort_csv_path, index=False)
print(f"\nCSV cohorte sauvegarde : {cohort_csv_path}")

# Export JSON de la cohorte
cohort_json_path = OUTPUT_DIR / "cohort_summary.json"
cohort_df.to_json(cohort_json_path, orient="records", indent=2, force_ascii=False)
print(f"JSON cohorte sauvegarde : {cohort_json_path}")


In [ ]:
## Cellule 10 — Visualisation statistique de la cohorte

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Cohorte MRD Triple — Distribution des valeurs MRD", fontsize=14, fontweight="bold")

LOQ = PARAMS.get("lod_loq_loq", 0.005)
LOD = PARAMS.get("lod_loq_lod", 0.009)

col_labels = [
    ("MRD_method_1_pct", "M1 : delta Metaclusters", "#4f86c6"),
    ("MRD_method_2_pct", "M2 : Distance euclidienne MST", "#e88a3e"),
    ("MRD_method_3_pct", "M3 : Mapping Populations", "#5caa5e"),
]

# Ligne 1 : histogrammes des valeurs MRD
for ax, (col, title, color) in zip(axes[0], col_labels):
    if col not in cohort_df.columns:
        ax.text(0.5, 0.5, "N/A", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title)
        continue
    valid = cohort_df[col].dropna()
    ax.hist(valid, bins=max(5, len(valid)//3), color=color, edgecolor="white", alpha=0.85)
    ax.axvline(LOQ, color="red",    linestyle="--", linewidth=1.3, label=f"LOQ {LOQ}%")
    ax.axvline(LOD, color="orange", linestyle=":",  linewidth=1.3, label=f"LOD {LOD}%")
    ax.set_title(title)
    ax.set_xlabel("MRD (%)")
    ax.set_ylabel("Nb patients")
    ax.legend(fontsize=8)

# Ligne 2 : comparaison des 3 methodes par patient (grouped bar)
ax_compare = axes[1, 0]
if all(c in cohort_df.columns for c in ["MRD_method_1_pct", "MRD_method_2_pct", "MRD_method_3_pct"]):
    x = np.arange(len(cohort_df))
    w = 0.25
    ax_compare.bar(x - w,   cohort_df["MRD_method_1_pct"], w, color="#4f86c6", label="M1")
    ax_compare.bar(x,       cohort_df["MRD_method_2_pct"], w, color="#e88a3e", label="M2")
    ax_compare.bar(x + w,   cohort_df["MRD_method_3_pct"], w, color="#5caa5e", label="M3")
    ax_compare.axhline(LOQ, color="red", linestyle="--", linewidth=1.2)
    ax_compare.set_xticks(x)
    ax_compare.set_xticklabels(cohort_df["patient_id"].tolist(), rotation=45, ha="right", fontsize=8)
    ax_compare.set_title("Comparaison M1 / M2 / M3 par patient")
    ax_compare.set_ylabel("MRD (%)")
    ax_compare.legend()

# Ligne 2 : meilleure methode (camembert)
ax_pie = axes[1, 1]
if "best_method" in cohort_df.columns:
    method_counts = cohort_df["best_method"].value_counts()
    ax_pie.pie(method_counts.values, labels=method_counts.index,
               autopct="%1.0f%%", startangle=90,
               colors=["#4f86c6", "#e88a3e", "#5caa5e", "#aaa", "#ccc"])
    ax_pie.set_title("Repartition meilleure methode")

# Ligne 2 : taux de divergence (barres)
ax_div = axes[1, 2]
if "divergence_flag" in cohort_df.columns:
    n_div    = cohort_df["divergence_flag"].sum()
    n_nodiv  = len(cohort_df) - n_div
    ax_div.bar(["Convergent", "Divergent"], [n_nodiv, n_div],
               color=["#5caa5e", "#e05555"], edgecolor="white")
    ax_div.set_title("Convergence / Divergence inter-methodes")
    ax_div.set_ylabel("Nb patients")
    for idx, val in enumerate([n_nodiv, n_div]):
        ax_div.text(idx, val + 0.1, str(val), ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
## Cellule 11 — Tableau de suivi longitudinal (si plusieurs timepoints par patient)
#
# Si les FCS sont nommes selon le pattern : PatientID_YYYY-MM-DD.fcs
# ou PatientID_TP1.fcs / PatientID_TP2.fcs
# cette cellule regroupe les valeurs MRD par patient et les trace en courbe.

import re

def extract_patient_timepoint(patient_id: str):
    """Extrait patient_base et timepoint depuis le nom de fichier."""
    # Exemples : "Pat01_2024-01-15" ou "Pat01_D0" ou "Pat01_TP1"
    m = re.match(r"^(.+?)[-_](D\d+|TP\d+|\d{4}-\d{2}-\d{2}|\d+)$", patient_id, re.IGNORECASE)
    if m:
        return m.group(1), m.group(2)
    return patient_id, "T0"  # Pas de pattern reconnu -> patient unique sans timepoint

if "patient_id" in cohort_df.columns:
    cohort_df[["patient_base", "timepoint"]] = cohort_df["patient_id"].apply(
        lambda pid: pd.Series(extract_patient_timepoint(pid))
    )

    longitudinal_patients = cohort_df[cohort_df.duplicated("patient_base", keep=False)]["patient_base"].unique()

    if len(longitudinal_patients) > 0:
        print(f"Patients avec suivi longitudinal detectes : {list(longitudinal_patients)}")

        fig, ax = plt.subplots(figsize=(12, 5))
        for pat in longitudinal_patients:
            subset = cohort_df[cohort_df["patient_base"] == pat].sort_values("timepoint")
            if "MRD_method_1_pct" in subset.columns:
                ax.plot(subset["timepoint"], subset["MRD_method_1_pct"],
                        marker="o", label=f"{pat} M1")
        ax.axhline(LOQ, color="red",    linestyle="--", linewidth=1.2, label=f"LOQ {LOQ}%")
        ax.axhline(LOD, color="orange", linestyle=":",  linewidth=1.2, label=f"LOD {LOD}%")
        ax.set_title("Suivi longitudinal MRD (Methode 1)")
        ax.set_xlabel("Timepoint")
        ax.set_ylabel("MRD (%)")
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("Pas de timepoints detectes dans les noms de fichiers.")
        print("Convention attendue : PatientID_D0.fcs, PatientID_TP1.fcs, PatientID_2024-03-15.fcs")
else:
    print("cohort_df non disponible — executer d'abord la cellule 8.")

## Section 4 — Utilitaires et débogage

Cellules optionnelles pour inspecter le modèle NBM, diagnostiquer les prétraitements et visualiser la grille SOM.

In [ ]:
## Cellule 12 — Inspection du modele NBM (heatmap MFI)

import matplotlib.pyplot as plt
import numpy as np

# Heatmap des profils d'expression par noeud SOM
fig, ax = plt.subplots(figsize=(max(14, len(nbm.markers) * 0.5), 6))
node_mfi_df = pd.DataFrame(nbm.node_mfi, columns=nbm.markers)

# Normalisation par marqueur pour la lisibilite (z-score)
node_mfi_norm = (node_mfi_df - node_mfi_df.mean()) / (node_mfi_df.std() + 1e-9)

im = ax.imshow(node_mfi_norm.T, aspect="auto", cmap="RdYlBu_r", vmin=-2, vmax=2)
ax.set_yticks(range(len(nbm.markers)))
ax.set_yticklabels(nbm.markers, fontsize=8)
ax.set_xlabel("Noeud SOM")
ax.set_title("Heatmap MFI normalisee (z-score) par noeud SOM — Modele NBM")
plt.colorbar(im, ax=ax, label="z-score MFI")

# Superposer les metaclusters (bandes verticales)
mc_ids = nbm.metaclusters
unique_mc = np.unique(mc_ids)
cmap_mc = plt.cm.get_cmap("tab10", len(unique_mc))
for mc in unique_mc:
    nodes_mc = np.where(mc_ids == mc)[0]
    if len(nodes_mc) > 0:
        ax.axvline(nodes_mc[0] - 0.5, color=cmap_mc(int(mc) % 10),
                   linewidth=1.5, linestyle="--", alpha=0.5)
        ax.text(nodes_mc.mean(), -0.7, f"MC{mc}", ha="center",
                fontsize=7, color=cmap_mc(int(mc) % 10), fontweight="bold")

plt.tight_layout()
plt.show()

print(f"Modele NBM : {nbm.node_mfi.shape[0]} noeuds x {len(nbm.markers)} marqueurs")
print(f"Metaclusters : {dict(zip(*np.unique(mc_ids, return_counts=True)))}")

In [ ]:
## Cellule 13 — Diagnostic preprocessing d'un FCS

# Charge un FCS brut et affiche les statistiques de QC avant / apres preprocessing
fcs_diag = fcs_files_patient[0]  # Modifier le fichier si besoin

import flowio, numpy as np

# Lecture brute
with open(fcs_diag, "rb") as f:
    raw = flowio.FlowData(f)

channels = [raw.channels[k]["PnN"] for k in sorted(raw.channels.keys(), key=int)]
data_raw  = np.array(list(raw.events)).reshape(-1, len(channels))

print(f"FCS : {fcs_diag.name}")
print(f"Canaux bruts : {len(channels)} | Cellules brutes : {data_raw.shape[0]:,}")
print(f"Canaux : {channels}")

# Preprocessing
from mrd_pipeline import preprocess_fcs
X_proc, markers_proc, qc = preprocess_fcs(fcs_diag, PARAMS)
print(f"\nApres preprocessing :")
print(f"  Cellules retenues : {X_proc.shape[0]:,}")
print(f"  Marqueurs retenus : {len(markers_proc)}")
print(f"  Marqueurs : {markers_proc}")
print(f"  QC : {qc}")

# Verifier l'absence de NaN
n_nan = int(np.isnan(X_proc).sum())
if n_nan > 0:
    print(f"\n  ATTENTION : {n_nan} valeurs NaN detectees dans la matrice preprocessee !")
else:
    print(f"\n  OK : aucun NaN dans la matrice preprocessee.")

---

## Notes méthodologiques et références ELN 2022

### Seuils MRD (ELN 2022)
| Paramètre | Valeur | Description |
|-----------|--------|-------------|
| **LOD** | 0.009 % | Limite de détection — en dessous = non quantifiable |
| **LOQ** | 0.005 % | Limite de quantification |
| **Fold-change** | × 2 | Seuil M1 : proportion patient > 2× NBM = positif |
| **Min events/node** | ≥ 17 | Minimum ELN par nœud SOM pour être comptabilisé |
| **Fold-change NBM** | > 1.9 % | Fréquence maximale d'une population en NBM |

### Choix de la meilleure méthode (`select_best_mrd_method`)
- **M1** (delta MC) est préférée quand elle détecte des metaclusters avec FC > 2× et ≥ 17 événements.
- **M2** (distances) est un contrôle croisé : si bimodalité élevée → confirmation M1 ; si diverge → flag.
- **M3** (mapping pop.) est la méthode ELN de référence quand le `pop_mfi_ref` est disponible.
- Si les 3 méthodes divergent (écart > 2 ordres de grandeur), un **flag de divergence** est levé → revue manuelle.

### Format de la table Kaluza
La table CSV générée (`export_kaluza_csv`) contient une ligne par nœud SOM avec :
- `node_id`, `metacluster`, `n_cells`, `pct_of_total`
- `mrd_m1_contributing` — nœud contribuant à la MRD selon M1 (boolean)
- `mrd_m2_deviant` — nœud déviant selon M2 (boolean)
- `mrd_m3_unknown` — nœud Unknown selon M3 (boolean)
- MFI de chaque marqueur (colonnes supplémentaires)

### Références
- Schuurhuis GJ et al. *ELN MRD recommendations*, Blood 2018.
- Döhner H et al. *ELN 2022 recommendations for AML*, Blood 2022.
- Van Gassen S et al. *FlowSOM*, Cytometry A 2015.
- LSCflow ALFA protocol — Tube 1 LAIP, Tube 2 LSC, Tube 3 Monocytes.